In [1]:
# packages to input for analysis
import skimage as sk
import os
import numpy as np
import pandas as pd
import re
from cellpose import models, io
from random import randint
from glob import glob
import napari

In [2]:
def get_center_project(file, path, name):
    """
    Get the center 3 slices and create a median projection
    Additionally apply small median filter determined by shape of image
    Returns the median projected center slices, nuclei channel (from projection), and median filter processed
    """
    img = sk.io.imread(file)
    center_z = img.shape[1]//2
    mid_slices = img[:,center_z-1:center_z+2,...]
    median_proj = np.median(mid_slices,axis=1)
    sk.io.imsave(os.path.join(path,f"{name}_center_median_proj.tiff"),median_proj,check_contrast=False)
    if img.shape[-1] == 1024:
        md_gauss = sk.filters.gaussian(median_proj,2)
    else:
        md_gauss = sk.filters.gaussian(median_proj,1)
    return median_proj, md_gauss, median_proj[0,...]

def process_nuc_channel(nuc_channel):
    """
    Clean the background and cytoplasm signal observed in some images using gaussian division and tophat filter with image size dependent footprint
    Returns the tophat filtered image with a gaussian filter applied for better nuclei segmenation in CellPose
    """
    sigma = nuc_channel.shape[-1]//3
    gauss = sk.filters.gaussian(nuc_channel,sigma)
    norm = sk.exposure.rescale_intensity(nuc_channel / gauss,out_range=(0,256))
    if nuc_channel.shape[-1] == 1024:
        sm_gauss = sk.filters.gaussian(norm,sigma=1)
        footprint = sk.morphology.disk(14)
        top_hat = sk.morphology.white_tophat(sm_gauss,footprint)
        md_gauss = sk.filters.gaussian(top_hat,4)
    else:
        sm_gauss = sk.filters.gaussian(norm,sigma=0.5)
        footprint = sk.morphology.disk(7)
        top_hat = sk.morphology.white_tophat(sm_gauss,footprint)
        md_gauss = sk.filters.gaussian(top_hat,2)        
    return md_gauss

def run_cp(img,cell_prob=0.0,flow=0.4,size=15):
    """ 
    Get cell or nuclei segmentation masks using CellPose4
    1024x1024 images required rescaling to get objects closer to preferred 30 px diameter
    Objects are resampled on the full size image to get better outlines
    Returns masks as label image
    """
    if img.shape[-1] == 1024:
        masks, _, _ = model.eval(img, rescale=0.5, resample=True, normalize=True, flow_threshold=flow, cellprob_threshold=cell_prob,min_size=size)
    else:
        masks, _, _ = model.eval(img, normalize=True, flow_threshold=flow, cellprob_threshold=cell_prob,min_size=size)
    return masks

def _filter_cells(cleared_cells,relabled_nuc):
    """ 
    Cells without matched nuclei segmentation from CellPose are removed
    Uses region properties to look for cells with a max value of 0 (nuclei label image is treated as an intensity image, a max value of 0 means there are no nuclei labels present within the cell)
    All cells without a nuclei are relabeled to a value of 0 (background)
    Returns the filtered cell labels
    """
    cell_props = sk.measure.regionprops(cleared_cells,relabled_nuc)
    cells_no_nuc = []
    cell_indicies = np.arange(np.max(cleared_cells)+1)
    for c in cell_props:
        label = c.label
        if c.intensity_max == 0:
            cells_no_nuc.append(label)
    cells_to_mask = np.isin(cell_indicies, cells_no_nuc)
    cells_mask = cells_to_mask[cleared_cells.reshape(-1)].reshape(cleared_cells.shape)
    cleared_cells[cells_mask] = 0
    relabled_cells, _, _ = sk.segmentation.relabel_sequential(cleared_cells)
    return relabled_cells

def clear_object_boarders(cell_masks,nuc_masks,name,path):
    """
    Remove all objects touching the image border, any cells without nuclei, and cleans the segmentation so that all nuclei are fully contained within the cell
    All cell and nuclei objects are relabed to have matching label IDs and label images are saved 
    """
    cleared_cells, _, _ = sk.segmentation.relabel_sequential(sk.segmentation.clear_border(cell_masks))
    binary_cells = np.array(cleared_cells > 0,dtype="bool")
    masked_nuc = nuc_masks * binary_cells
    masked_nuc[masked_nuc>0] = 1
    matched_cells_to_nuc = _filter_cells(cleared_cells,masked_nuc)
    cell_bounaries = sk.segmentation.find_boundaries(matched_cells_to_nuc)
    masked_boundaries = np.array(matched_cells_to_nuc > 0,dtype='bool')
    masked_boundaries[cell_bounaries] = False
    if cell_masks.shape[-1] == 1024:
        footprint = sk.morphology.disk(2)
    else:
        footprint = sk.morphology.disk(1)
    eroded_masked_boundaries = sk.morphology.erosion(masked_boundaries,footprint)
    relabeled_nuc = (masked_nuc * eroded_masked_boundaries)*matched_cells_to_nuc # using eroded cells to crop nuclei within the cell boundary and minimize/remove overlaping nuclei labels within neighboring cells
    sk.io.imsave(os.path.join(path,f"{name}_cell_masks.tif"),matched_cells_to_nuc,check_contrast=False)
    sk.io.imsave(os.path.join(path,f"{name}_nuclei_masks.tif"),relabeled_nuc,check_contrast=False)
    return matched_cells_to_nuc, relabeled_nuc


def cyto_objects(cells, nuc, name,path):
    """
    Cytoplasm objects are generated by subtracking the nuclei label array from the cell label array
    Because all cells and nuclei have same label value, pixels of overlapping cell and nuclei labels are set to 0 (background)
    Returns the result 
    """
    label_cyto = cells - nuc
    sk.io.imsave(os.path.join(path,f"{name}_cyto_masks.tif"),label_cyto,check_contrast=False)
    return label_cyto

def rename(x):
    """ 
    Create a python friendly version of the image name for easier sorting 
    """
    pattern_1 = re.compile(r"^(?P<cell>....)_(?P<condition>DMSO|drug)_(?P<time_point>\d+)_")
    pattern_2 = re.compile(r"^(?P<cell>....)_(?P<time_point>\d+)\w+_(?P<condition>DMSO|drug)_")
    if re.match(pattern_1,x):
        return f"{re.search(pattern_1,x).group('cell').lower()}_{re.search(pattern_1,x).group('condition').lower()}_{re.search(pattern_1,x).group('time_point')}min"
    elif re.match(pattern_2,x):
        return f"{re.search(pattern_2,x).group('cell').lower()}_{re.search(pattern_2,x).group('condition').lower()}_{re.search(pattern_2,x).group('time_point')}min"
    else:
        return "No_Match"

def cell_type(x):
    """ 
        Add in a column for the cell type for easier sorting
    """
    if 'hela' in x:
        return 'hela'
    if 'pc12' in x:
        return 'pc12'

def condition(x):
    """ 
        Add in a column for the treatment group for easier sorting
    """
    if 'drug' in x:
        return 'drug'
    if 'dmso' in x:
        return 'dmso'

def get_measurements(label_cyto,label_cells,label_nuc,img,name):
    """ 
        Collect intensity measurements in all compartments and then calculate the ratio of the nuclear intensity and cytoplasm intensity
        Returns a merged dataframe of measurements from all comaprtments
    """    
    props = ["label","intensity_mean","intensity_median"]
    reshape_img = np.moveaxis(img,0,-1)
    cyto_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(label_cyto,reshape_img,properties=props))
    nuc_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(label_nuc,reshape_img,properties=props))
    cell_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(label_cells,reshape_img,properties=props))
    cyto_props = cyto_props.add_prefix("Cyto_")
    nuc_props = nuc_props.add_prefix("Nuc_")
    cell_props = cell_props.add_prefix("Cell_")
    cyto_props["Nur77_Nuc_Cyto_Ratio_Mean"] = nuc_props["Nuc_intensity_mean-1"]/cyto_props["Cyto_intensity_mean-1"]
    cyto_props["Nur77_Nuc_Cyto_Ratio_Median"] = nuc_props["Nuc_intensity_median-1"]/cyto_props["Cyto_intensity_median-1"]
    merged = pd.concat([cell_props,nuc_props,cyto_props],axis=1)
    merged['Image'] = name
    merged['Clean_Name'] = merged['Image'].apply(rename)
    merged['Cell_Type'] = merged['Clean_Name'].apply(cell_type)
    merged['Treatment'] = merged['Clean_Name'].apply(condition)
    return merged


In [ ]:
#Batch analysis set up:
files = sorted(glob("/DISKS/2TB/gkristin/Nur77_NR4A1_Translocation/Exported_Tiffs/*.tif"))
names = [os.path.basename(f)[:-4] for f in files]
model = models.CellposeModel(gpu=True)
projections_save_path = "/DISKS/2TB/gkristin/Nur77_NR4A1_Translocation/Center_Projections/"
masks_save_path = "/DISKS/2TB/gkristin/Nur77_NR4A1_Translocation/Output_Masks/"
measurements_path = "/DISKS/2TB/gkristin/Nur77_NR4A1_Translocation/"
dfs = []
for f, n in zip(files, names):
    print(f"processing {n}")
    img = sk.io.imread(f)
    if len(img.shape) == 4:
        median_proj, median_filter, nuc_channel = get_center_project(f,projections_save_path,n)
        nuc_norm = process_nuc_channel(nuc_channel)
        cell_masks = run_cp(median_filter,flow=0.6,size=100)
        nuc_masks = run_cp(nuc_norm,cell_prob=-1.0,flow=0.6)
        cleared_cells, cleared_nuc = clear_object_boarders(cell_masks,nuc_masks,n,masks_save_path)
        cyto_masks = cyto_objects(cleared_cells, cleared_nuc,n,masks_save_path)
        df = get_measurements(cyto_masks,cleared_cells,cleared_nuc,median_proj,n)
        dfs.append(df)
    else:
        print(f"Skipping {n}, has shape {img.shape}")        
all_data = pd.concat(dfs,ignore_index=True) #merge all dataframes from all images analyzed to then save as a single file
all_data.to_csv(os.path.join(measurements_path,'Nur77_NR4A1_Translocation_Ratio_Measurements.csv'),index=False)

Plotting data functions below

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
all_data = pd.read_csv('/DISKS/2TB/gkristin/Nur77_NR4A1_Translocation/Nur77_NR4A1_Translocation_Ratio_Measurements.csv')

In [ ]:
per_img_mean = all_data.groupby(['Image', 'Clean_Name', 'Cell_Type','Treatment'],as_index=False).mean(numeric_only=True)

In [ ]:
per_img_mean.to_csv('/DISKS/2TB/gkristin/Nur77_NR4A1_Translocation/Nur77_NR4A1_Translocation_Ratio_Measurements_per_img_medians.csv',index=False)

In [ ]:
per_group_mean = per_img_mean.groupby(['Clean_Name','Cell_Type','Treatment'],as_index=False).mean(numeric_only=True)

In [ ]:
hela_groups = per_group_mean.query('Cell_Type == "hela"').sort_values('Clean_Name')
pc12_groups = per_group_mean.query('Cell_Type == "pc12"').sort_values('Clean_Name')

In [ ]:
import matplotlib.patches as mpatches

In [ ]:
print(sns.color_palette("pastel").as_hex())
['#a1c9f4', '#ffb482', '#8de5a1', '#ff9f9b', '#d0bbff', '#debb9b', '#fab0e4', '#cfcfcf', '#fffea3', '#b9f2f0'] # output from above line

In [ ]:
fig, axs = plt.subplots(2,1,figsize=(7,7))
sns.barplot(hela_groups,x='Clean_Name', y='Nur77_Nuc_Cyto_Ratio_Mean', hue='Treatment',order=['hela_dmso_30min','hela_dmso_60min','hela_dmso_120min','hela_drug_30min','hela_drug_60min','hela_drug_90min','hela_drug_120min'], legend=False, palette='pastel',ax=axs[0])
axs[0].set_title('Hela Cells')
axs[0].set_ylabel('Ratio (Mean Nuc IF/Mean Cyto IF)')
#axs[0].yaxis.set_major_formatter(FormatStrFormatter('%.3f'))
axs[0].tick_params('x',labelrotation=45, labelrotation_mode='xtick')
axs[0].xaxis.set_label_text('')
axs[0].set_xticklabels(['dmso 30min','dmso 60min','dmso 120min','drug 30min','drug 60min','drug 90min','drug 120min'])
#axs[0].spines.right.set_visible(False)
#axs[0].spines.top.set_visible(False)
sns.barplot(pc12_groups,x='Clean_Name', y='Nur77_Nuc_Cyto_Ratio_Mean', hue='Treatment',order=['pc12_dmso_30min','pc12_dmso_90min','pc12_drug_30min','pc12_drug_60min','pc12_drug_90min','pc12_drug_120min'], legend=False, palette='pastel', ax=axs[1])
axs[1].set_title('PC12 Cells')
axs[1].set_ylabel('Ratio (Mean Nuc IF/Mean Cyto IF)')
#axs[0].yaxis.set_major_formatter(FormatStrFormatter('%.3f'))
axs[1].tick_params('x',labelrotation=45, labelrotation_mode='xtick')
axs[1].xaxis.set_label_text('')
axs[1].set_xticklabels(['dmso 30min','dmso 90min','drug 30min','drug 60min','drug 90min','drug 120min'])
dmso_patch = mpatches.Patch(color='#a1c9f4', label='dmso')
treated_patch = mpatches.Patch(color='#ffb482', label='treated')
fig.legend(handles=[dmso_patch,treated_patch],bbox_to_anchor=(1.05, 1),loc='upper left')
plt.tight_layout()
plt.savefig('/DISKS/2TB/gkristin/Nur77_NR4A1_Translocation/Mean_Ratios.png',dpi=300)
